# Understanding 3D Tiles

[The 3D Tiles Standard](https://docs.ogc.org/cs/22-025r4/22-025r4.html) allows for streaming of large geospatial datasets. It is designed for digital twins of large cities where the triangle count as well as the amount of information is large. The high level way this works, is by streaming the data to the recipient instead of loading the full file into memory at once. This enables a smooth user experience regardless of where in the city they are. Let's explore a bit more on how this works and if we can derive any userful information from it.

From a high level read through of the standards document, we see that this standard utilizes a lot of techniques we have already explored in previous projects. LOD compression, octree subdivisions, glTF file formats. Seems like we're pretty close to making this happen. So lets jump in.

## Key Concepts

The standard makes multiple mentions of the HLOD concept, or Hierarchical Level of Detail modelling. This method stores the LOD's of a mesh within levels of an octree, thereby reducing the number of distance checks which need to be conducted. The standard also mentions that the metric used to determine when to switch LOD's is called the Screen Space Error, which measures in pixels if the tile is sufficiently rendered or if its children should be considered.

An important note mentioned in the standard is that of spatial coherence. A parent node's bounding volumne must contain all the geometry of its children. However, a child within the parent can have it's bounding volume exceed the bounds fo the parent. This is because the bounding volume of the child may contain empty space. However, the exact geometry of the children must be entirely within the parent to ensure that tree traversal properly captures these items.

The standard mentions certain key data formats which can be used in conjunction with 3D tiles- Batched 3d model, Instanced 3D model etc. This is worth another read. Section 7.

The standard mentions storing data using interleaving bytes, or Morton Codes. This will be something worth checking out and implementing well as it appears to significantly speed things up. Section 8.1.

This makes sense to me, and I'd like to take a first pass at creating something.

## Imports and Modules

In [1]:
import numpy as np
import pygltflib
import pymeshlab

## Brief Theory and Concept

In my head, this is what makes sense to me. We essentially have 2 versions of our LOD meshes- one that is low res and one that is high res. We wish to be able to store this data within a tree like data strcuture such that our querying sphere can quickly access it. Different levels of the tree will have different bounding volumes, and depending on which one is intersected by our query sphere, a different mesh will be active at any given time. In my head, low res meshes will be stored at higher levels of the tree while high res meshes get pushed down to the leaves.

In our [Octree working example](https://suryashch.github.io/octree/), we were able to visualize what an octree looks like. Let's take a look here.

![Octree Example](../img/octree-example.png)

In this example, we can clearly see the different levels of the octree. In theory, each one of these cubes should store references to the different objects that should be active in the scene at a given time. This way, if our query sphere intersects with the red cube, we immediately know to show all the objects that are stored in it. Let's try this out with a bigger model.

In [2]:
gltf = pygltflib.GLTF2().load('data/mep-fittings_hi-lo-res_ifc-guid-2.glb')

In theory, the bigger an object is on the screen, the higher up it should be stored in the octree. Since, long pipes will be visible from further away than something like a small valve. In this case, the longer / bigger the bounding volume, the higher up in the tree it will be. This exact relationship is to be determined.

Another note we can make here is that each level of the octree subdivides space into half. Therefore, going from the lowest level up, we can say with certainty how further each parent cube will be from the object- essentially our distance LOD function. If we want our object to be visible when the camera is 10 units away, we must take the leaf node corresponding to the object, and keep adding levels until we reach 10 units. Here, we will store the object. 

## Links

1) [The 3D Tiles Standard](https://docs.ogc.org/cs/22-025r4/22-025r4.html)